# 8_langsmith_izleri — Sunum İçin Üç Senaryonun LangSmith İzleri

Bu notebook, iki projenin Agent zincirini sırayla çalıştırıp her birini LangSmith'te **ayrı, adlandırılmış bir iz** olarak kaydediyor:

| # | Proje | Senaryo | Üst iz adı |
|---|---|---|---|
| 1 | PJM / AEP (ana proje) | Normal işletim | `1 - PJM normal` |
| 2 | PJM / AEP (ana proje) | Dar marj | `2 - PJM dar marj` |
| 3 | Türkiye (bonus) | Gün öncesi karar | `3 - TR gun oncesi` |

**İzler nereye gidiyor:** Üçü de tek bir sunum projesine gidiyor: `sunum-izleri`. Önceki çalıştırmaların izleri kendi projelerinde (`bitirme-proje-grid`, `tr-grid-agent`) kalıyor ve bunlarla karışmıyor.
- Her senaryo bir **üst iz** altında toplanıyor. İçinde RAG getirme adımı ve Gemini çağrısı ayrı alt adımlar olarak görünüyor.
- Her iz, `proje-1` / `proje-2` ve senaryo adıyla **etiketleniyor.**
- Notebook sonunda her izin doğrudan bağlantısı yazdırılıyor.

**İzleme yalnızca bu notebook'ta açılıyor** (`LANGSMITH_TRACING=true` aşağıda ayarlanıyor). `.env`'yi değiştirmeye gerek yok.

**Maliyet:** Toplam 3 Gemini çağrısı ve birkaç embedding isteği.

**Gizlilik:** PJM girdileri kişisel bilgi içermiyor. Türkiye zinciri, e-posta ve IP adreslerini LangSmith'e gönderilmeden önce maskeliyor (`tr_agent_zinciri`).

## A. Kurulum

- **Yollar:** İki projenin klasörleri ayrı tanımlanıyor; iki projenin modülleri de içe aktarılıyor. Dosya adları çakışmıyor: PJM'de `rag_retrieval`, `prompt_v1`; Türkiye'de `tr_rag_retrieval`, `tr_agent_zinciri`.
- **Anahtarlar:** `GEMINI_API_KEY` ve `LANGSMITH_API_KEY` Türkiye projesinin `.env`'sinden okunuyor.
- **Proje yönlendirmesi:** `AYRI_PROJELER = True` yapılırsa izler kendi projelerine gider (PJM → `bitirme-proje-grid`, Türkiye → `tr-grid-agent`).

In [1]:
import os
import sys
import json
import uuid
import pandas as pd
from dotenv import load_dotenv

KOK_PJM = r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje"
KOK_TR = r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje_Bonus"
load_dotenv(os.path.join(KOK_TR, "notepads", ".env"), override=True)
os.environ["LANGSMITH_TRACING"] = "true"               # yalnizca bu notebook icin

SUNUM_PROJESI = "sunum-izleri"
AYRI_PROJELER = False                                  # True: PJM -> bitirme-proje-grid, TR -> tr-grid-agent
PROJE_PJM = "bitirme-proje-grid" if AYRI_PROJELER else SUNUM_PROJESI
PROJE_TR = "tr-grid-agent" if AYRI_PROJELER else SUNUM_PROJESI

for yol in [os.path.join(KOK_PJM, "notepads"), os.path.join(KOK_TR, "notepads")]:
    if yol not in sys.path:
        sys.path.insert(0, yol)

from google import genai
from google.genai import types, errors
from langsmith import Client, traceable, tracing_context
from langsmith.run_helpers import get_current_run_tree

eksik = [k for k in ["GEMINI_API_KEY", "LANGSMITH_API_KEY"] if not os.getenv(k)]
assert not eksik, f".env'de eksik anahtar: {eksik}"
ls_istemci = Client()
gemini = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
IZLER = []        # (senaryo, proje, iz nesnesi, ozet)
print("Izler su projeye gidecek -> PJM:", PROJE_PJM, "| TR:", PROJE_TR)

Izler su projeye gidecek -> PJM: sunum-izleri | TR: sunum-izleri


## B. PJM Ana Projesinin Agent Zinciri

PJM projesindeki bileşenler **değiştirilmeden** kullanılıyor:
- `prompt_v1.SYSTEM_PROMPT`
- `rag_retrieval.retrieve_context` ve `build_rag_query`
- `get_agent_decision`: `1.ipynb`'deki tanımla aynı; `gemini-3.7-flash`, sıcaklık 0,2, JSON çıktı, iz adı `grid_agent_decision_with_rag`

**Senaryo girdisi:** PJM projesindeki tarihsel senaryo yöntemi (`tarihi_senaryo`):
- **Talep:** O saatin gerçekleşen talebi (geriye dönük, "hindsight").
- **Güven aralığı:** XGBoost'un doğrulanmış RMSPE'si (0,0097) ile.
- **Kapasite:** O saatten önceki 30 günün zirvesi × 1,20 (PJM'in resmî %20 IRM değeri).

RAG getirme adımı, izde ayrı görünmesi için ince bir izlenen sarmalayıcıyla çağrılıyor; davranışı değişmiyor.

In [2]:
from prompt_v1 import SYSTEM_PROMPT
import rag_retrieval as pjm_rag

@traceable(name="grid_agent_decision_with_rag", metadata={"forecast_model": "XGBoost"})
def get_agent_decision(content, system_prompt):
    response = gemini.models.generate_content(
        model="gemini-3.7-flash", contents=content,
        config=types.GenerateContentConfig(system_instruction=system_prompt, temperature=0.2,
                                           response_mime_type="application/json"))
    return response.text

@traceable(name="pjm_rag_retrieval")
def pjm_getir(sorgu):
    return pjm_rag.retrieve_context(sorgu)

aep = pd.read_csv(os.path.join(KOK_PJM, "data", "AEP_hourly.csv"), parse_dates=["Datetime"])
aep = aep.sort_values("Datetime").drop_duplicates("Datetime").reset_index(drop=True)
XGB_RMSPE = 0.0097

def pjm_girdi(tarih, kapasite_carpani=1.0):
    t = pd.Timestamp(tarih)
    satir = aep[aep["Datetime"] == t]
    assert len(satir), f"{t} AEP verisinde yok"
    mw = float(satir["AEP_MW"].iloc[0])
    once = aep[(aep["Datetime"] < t) & (aep["Datetime"] >= t - pd.Timedelta(days=30))]
    kapasite = round(once["AEP_MW"].max() * 1.20 * kapasite_carpani)
    h = t.hour
    donem = "day" if 7 <= h < 18 else ("peak" if 18 <= h < 23 else "night")
    kaynak = f"historical_{t:%Y-%m-%d_%H}_trailing30d_x1.20" + (f"_x{kapasite_carpani}" if kapasite_carpani != 1.0 else "")
    talep = {"region": "AEP", "forecast_mw": round(mw), "confidence_interval": [round(mw * (1 - XGB_RMSPE)), round(mw * (1 + XGB_RMSPE))]}
    durum = {"available_capacity_mw": kapasite, "time_period": donem, "capacity_source": kaynak}
    return talep, durum

def pjm_zincir(tarih, kapasite_carpani=1.0):
    talep, durum = pjm_girdi(tarih, kapasite_carpani)
    if hasattr(pjm_rag, "build_rag_query"):
        sorgu = pjm_rag.build_rag_query(talep["forecast_mw"], durum["available_capacity_mw"], durum["time_period"])
    else:
        sorgu = f"grid capacity shortfall reserve activation {durum['time_period']}"
    baglam = pjm_getir(sorgu)
    content = (f"<demand_forecast>\n{json.dumps(talep)}\n</demand_forecast>\n<grid_status>\n{json.dumps(durum)}\n</grid_status>\n"
               f"<official_procedure_context>\n{baglam}\n</official_procedure_context>")
    karar = json.loads(get_agent_decision(content, SYSTEM_PROMPT))
    marj = (durum["available_capacity_mw"] - talep["forecast_mw"]) / durum["available_capacity_mw"]
    return {"tarih": str(pd.Timestamp(tarih)), "talep_mw": talep["forecast_mw"], "kapasite_mw": durum["available_capacity_mw"],
            "marj": round(marj, 4), "donem": durum["time_period"], "rag_sorgusu": sorgu,
            "status": karar.get("status"), "decision": karar.get("decision"), "karar_json": karar}

def calistir(senaryo, proje, etiketler, fonk, *args, **kwargs):
    """Senaryoyu, verilen LangSmith projesinde ve etiketlerle tek bir ust iz altinda calistirir."""
    @traceable(name=senaryo)
    def ust(*a, **k):
        IZLER.append([senaryo, proje, get_current_run_tree(), None])
        return fonk(*a, **k)
    with tracing_context(project_name=proje, tags=etiketler, metadata={"sunum": "bitirme", "senaryo": senaryo}):
        try:
            sonuc = ust(*args, **kwargs)
        except errors.ClientError as e:
            if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
                print("DURDU: Gemini kotasi doldu."); return None
            raise
    IZLER[-1][3] = sonuc
    return sonuc
print("PJM bilesenleri yuklendi | build_rag_query var mi:", hasattr(pjm_rag, "build_rag_query"), "| AEP:", aep["Datetime"].min(), "->", aep["Datetime"].max())

PJM bilesenleri yuklendi | build_rag_query var mi: True | AEP: 2004-10-01 01:00:00 -> 2018-08-03 00:00:00


### B1. Senaryo 1: PJM, normal işletim

**10 Aralık 2009, saat 03:00 (gece).** Talebin düşük olduğu saat. 30 günlük zirve × 1,20 kapasiteyle geniş bir marj beklenir; beklenen karar `normal_operation`.

In [3]:
s1 = calistir("1 - PJM normal", PROJE_PJM, ["proje-1", "pjm", "normal"], pjm_zincir, "2009-12-10 03:00")
if s1:
    print(f"Talep {s1['talep_mw']} MW | kapasite {s1['kapasite_mw']} MW | marj %{s1['marj'] * 100:.1f} | donem {s1['donem']}")
    print("Karar:", s1["status"], "/", s1["decision"])
    print(json.dumps(s1["karar_json"], ensure_ascii=False, indent=2)[:1500])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Talep 15749 MW | kapasite 23717 MW | marj %33.6 | donem night
Karar: normal / normal_operation
{
  "reasoning_steps": "Forecasted demand of 15,749 MW is well below available capacity of 23,717 MW, providing a substantial capacity cushion (~33.6% surplus). Occurring during the off-peak night period, reserve requirements are comfortably satisfied.",
  "status": "normal",
  "decision": "normal_operation",
  "justification": "Operating reserves substantially exceed standard PJM Primary and Secondary Reserve Requirements under normal off-peak night conditions, requiring no supplemental market or emergency actions.",
  "uncertainty": "low",
  "confidence_note": "Data is complete with consistent interval bounds; forecast upper limit (15,902 MW) remains well below available capacity."
}


### B2. Senaryo 2: PJM, dar marj

**10 Aralık 2009, saat 18:00 (akşam zirvesi).** PJM projesindeki tarihsel tarama, bu saati 30 günlük geçmişine göre en olağan dışı yüksek talep anlarından biri olarak bulmuştu: talep 21.072 MW, önceki 30 günün zirvesinin 1,059 katı. ×1,20 kapasite formülüyle marj yaklaşık **%11,8**, yani IRM'in (%20) altında.

`KAPASITE_CARPANI` 1'in altına indirilirse (örneğin 0,95), kapasite yapay olarak daha da düşürülür ve rezerv kararı tetiklenebilir.

In [4]:
KAPASITE_CARPANI = 1.0
s2 = calistir("2 - PJM dar marj", PROJE_PJM, ["proje-1", "pjm", "dar-marj"], pjm_zincir, "2009-12-10 18:00", KAPASITE_CARPANI)
if s2:
    print(f"Talep {s2['talep_mw']} MW | kapasite {s2['kapasite_mw']} MW | marj %{s2['marj'] * 100:.1f} | donem {s2['donem']}")
    print("Karar:", s2["status"], "/", s2["decision"])
    print(json.dumps(s2["karar_json"], ensure_ascii=False, indent=2)[:1500])

Talep 21072 MW | kapasite 23875 MW | marj %11.7 | donem peak
Karar: attention / demand_response_alert
{
  "reasoning_steps": "Forecasted demand of 21,072 MW against an available capacity of 23,875 MW leaves an operating reserve margin of approximately 13.3%, which falls below the typical 15-20% IRM threshold. Given that this occurs during the high-demand 'peak' period, the thin capacity margin requires proactive operational vigilance.",
  "status": "attention",
  "decision": "demand_response_alert",
  "justification": "Operating reserve margin is approximately 13.3% during peak hours; per PJM procedures, issuing a Maximum Generation Emergency / Load Management Alert provides early notification when system conditions may require demand response or emergency procedures.",
  "uncertainty": "low",
  "confidence_note": "Data is complete with clear confidence intervals and capacity metrics provided."
}


## C. Senaryo 3: Türkiye Bonus Projesinin Zinciri

`tr_agent_zinciri.Zincir` ile **7 Kasım 2025** yeniden oynatılıyor. Bu gün, yaz dışı bir akşam darlığı örneği: 6_agent_tr'de en dar saat 18:00'di ve karar `demand_response_alert` idi.

**İzin içinde görünenler:**
- `tr_mevzuat_retrieval`: Türkiye mevzuatından getirilen maddeler.
- `tr_agent_karar_aciklamasi`: Gemini çağrısı; e-posta/IP maskelenmiş.

**Not:** Üretim modeli (`3d_uretim_...`) bu günü eğitimde gördü. Bu iz, zincirin **nasıl çalıştığını** gösteriyor; bir doğruluk ölçüsü değil.

`CANLI = True` yapılırsa, geçmiş gün yerine **yarın** için canlı çalıştırma yapılır: EPİAŞ ve Open-Meteo'dan veri alınır, ve sonuç `data/canli/` klasörüne kaydedilmez.

In [5]:
from tr_agent_zinciri import Zincir, tekrar_oynatma_verisi
CANLI = False
TR_GUN = "2025-11-07"

def tr_zincir(gun):
    z = Zincir(KOK_TR)
    if CANLI:
        import tr_veri_toplayici as tv
        ham, bilgi = tv.canli_veri(KOK_TR, gun, z.meta_secilen)
        z.veri(ham); r = z.calistir(gun, kaynak_bilgisi=bilgi)
    else:
        z.veri(tekrar_oynatma_verisi(KOK_TR, z.meta_secilen)); r = z.calistir(gun)
    o, c = r["ozet"], r["cikti"]
    return {"gun": o["gun"], "model": f"{o['model']} ({z.surum})", "en_dar_saat": o["en_dar_saat"], "marj": o["en_dusuk_marj"],
            "status": c["durum"], "decision": c["karar"], "oncelik": c["oncelik"], "ozet": c["ozet"],
            "atiflar": [f"{a['belge']} | {a['madde']}" for a in c["mevzuat_atiflari"]], "dogrulama_notlari": r["notlar"]}

gun = (pd.Timestamp.now(tz="Europe/Istanbul").normalize() + pd.Timedelta(days=1)).strftime("%Y-%m-%d") if CANLI else TR_GUN
s3 = calistir("3 - TR gun oncesi", PROJE_TR, ["proje-2", "turkiye", "canli" if CANLI else "yeniden-oynatma"], tr_zincir, gun)
if s3:
    print(json.dumps(s3, ensure_ascii=False, indent=2))

{
  "gun": "2025-11-07",
  "model": "Hibrit (oran) + Open-Meteo (3d_uretim_20261002_1617)",
  "en_dar_saat": 18,
  "marj": 0.1657,
  "status": "attention",
  "decision": "demand_response_alert",
  "oncelik": "yuksek",
  "ozet": "2025-11-07 günü için yapılan gün öncesi değerlendirmesinde, saat 18:00'de marjın %16.57 seviyesine gerileyeceği öngörüldüğünden yüksek öncelikli talep tarafı katılımı uyarısı (demand_response_alert) kararı verilmiştir.",
  "atiflar": [
    "Elektrik Piyasası Dengeleme ve Uzlaştırma Yönetmeliği | MADDE 67/A",
    "Elektrik Piyasası Dengeleme ve Uzlaştırma Yönetmeliği | MADDE 72"
  ],
  "dogrulama_notlari": []
}


## D. İzlerin Bağlantıları ve LangSmith'te Nasıl Gösterilir

İzler LangSmith'e arka planda gönderiliyor. `flush()`, notebook bağlantıları yazdırmadan önce gönderimin bitmesini bekliyor.

**Sunumda gösterim:**
1. `smith.langchain.com` → **Projects** → `sunum-izleri`.
2. Liste en yeni iz üstte olacak şekilde sıralı. Üç iz: `1 - PJM normal`, `2 - PJM dar marj`, `3 - TR gun oncesi`.
3. Bir ize tıklayın. Sol tarafta adımlar görünür: RAG getirme ve Gemini çağrısı. Her adımın **Input** ve **Output** sekmesinde, Agent'a giden bağlam ve döndürdüğü JSON görünür.
4. Belirli bir projeyi filtrelemek için etiketlere tıklayın: `proje-1` veya `proje-2`.

In [6]:
ls_istemci.flush()
for c in {id(n[2].client): n[2].client for n in IZLER if n[2] is not None}.values():
    try:
        c.flush()
    except Exception:
        pass
satirlar = []
for senaryo, proje, iz, sonuc in IZLER:
    try:
        url = iz.get_url() if iz is not None else "(iz olusmadi)"
    except Exception as ex:
        url = f"(baglanti alinamadi: {str(ex)[:60]}; LangSmith'te {proje} projesine bakin)"
    satirlar.append({"senaryo": senaryo, "proje": proje,
                     "karar": (sonuc or {}).get("decision"), "marj_%": round((sonuc or {}).get("marj", float("nan")) * 100, 1),
                     "baglanti": url})
tablo = pd.DataFrame(satirlar)
with pd.option_context("display.max_colwidth", None):
    print(tablo.to_string(index=False))

          senaryo        proje                 karar  marj_%                                                                                                                                                                                                                                                   baglanti
   1 - PJM normal sunum-izleri      normal_operation    33.6 https://smith.langchain.com/o/a9a9f7dd-fdfa-4edc-9530-710dd716a369/projects/p/1f62f1e7-994a-4f11-bb73-0fc08042942b/trace/01a10c75-e29f-74e2-81e8-3db9f9d59425/run/01a10c75-e29f-74e2-81e8-3db9f9d59425?start_time=2026-10-05T14%3A26%3A54.751353%2B00%3A00
 2 - PJM dar marj sunum-izleri demand_response_alert    11.7 https://smith.langchain.com/o/a9a9f7dd-fdfa-4edc-9530-710dd716a369/projects/p/1f62f1e7-994a-4f11-bb73-0fc08042942b/trace/01a10c75-f3d8-7dd2-b852-1581676cd3f7/run/01a10c75-f3d8-7dd2-b852-1581676cd3f7?start_time=2026-10-05T14%3A26%3A59.160628%2B00%3A00
3 - TR gun oncesi sunum-izleri demand_response_alert    16.6 htt